In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import newton
from numpy import random
import os
import math

In [2]:
def solver(theta, tree, zcb, i, sigma, delta):
    #Create Pricing matrix for ZCBs
    price = np.zeros([i+2, i+2])
    #Assign last row to be payoff of zcb = 1
    price[:,len(price)-1] = 1

    #Assign New rate trees
    for row in range(0, i+1):
        if row == 0:
            tree[row,i] = tree[row,i-1] + theta*delta + sigma*math.sqrt(delta)
        else:
            tree[row,i] = tree[row-1,i-1] + theta*delta - sigma*math.sqrt(delta)

    #Pricing Tree iteration
    for col in reversed(range(0,i+1)):
        for row in range(0, col+1):
            rate = np.exp(-1*tree[row,col]*delta)
            price[row,col] = rate*0.5*(price[row,col+1] + price[row+1,col+1])

    return price[0,0] - zcb

In [3]:
class HoLeeModel:
    def __init__(self, zeros, sigma, delta):
        self.zeros = zeros
        self.sigma = sigma
        self.delta = delta

    def calibrate(self, tree, zcb, i):
        t0 = 0.5 #Initial guess for all theta
        theta = newton(solver, t0, args=(tree, zcb, i, self.sigma, self.delta))
        
        for row in range(0, i+1):
            if row == 0:
                tree[row,i] = tree[row, i-1] + theta*self.delta + self.sigma*math.sqrt(self.delta)
            else:
                tree[row,i] = tree[row, i-1] + theta*self.delta - self.sigma*math.sqrt(self.delta)
        
        return [theta, tree]
        
    def build(self):
        #Empty rate tree
        zcb = self.zeros
        tree = np.zeros([zcb.shape[1], zcb.shape[1]])
        #Empty theta Tree
        theta = np.zeros([zcb.shape[1]])

        #Initiate Zero Coupon rate
        tree[0,0] = -np.log(zcb[0,0])/self.delta

        r0 = tree[0,0]

        for i in range(1, len(theta)):
            solved = self.calibrate(tree, zcb[0,i], i)
            #Update array
            theta[i] = solved[0]
            tree = solved[1]
        return [r0, tree, theta]

    def rateTree(self):
        x = self.build()
        
        r0 = x[0]
        theta = x[2]
        tree = np.zeros([len(theta), len(theta)])
        tree[0,0] = r0
        
        for col in range(1, len(tree)):
            for row in range(0,col+1):
                if row == 0:
                    tree[row,col] = tree[row,col-1] + theta[col]*self.delta + self.sigma*math.sqrt(self.delta)
                else:
                    tree[row,col] = tree[row-1,col] - 2*self.sigma*math.sqrt(self.delta)

        return tree

In [4]:
class IRDerivativePricing:
    def __init__(self,rates,prob,delta,typ,notion,cpn, strike):
        self.rates = rates
        self.prob = prob
        self.delta = delta
        self.type = typ
        self.notion = notion
        self.cpn = cpn
        self.strike = strike

    def payoff(self):
        if self.type == 'bond':
            return self.notion
        else:
            return 0

    def cf_cap(self):
        cf = np.zeros([len(self.rates)+1, len(self.rates)+1])
        for col in range(0, len(cf)-1):
            for row in range(0, col+1):
                rate = self.rates[row,col]
                cf[row,col] = self.delta*self.notion*max(rate - self.strike/100, 0)
        return cf

    def cf_floor(self):
        cf = np.zeros([len(self.rates)+1, len(self.rates)+1])
        for col in range(0, len(cf)-1):
            for row in range(0, col+1):
                rate = self.rates[row,col]
                cf[row,col] = self.delta*self.notion*max(self.strike/100 - rate, 0)
        return cf
    
    def priceTree(self):
        tree = np.zeros([len(self.rates)+1, len(self.rates)+1])
        tree[:,len(tree)-1] = self.payoff()
        
        if self.type == "cap":
            cf = self.cf_cap()
        elif self.type == "floor":
            cf = self.cf_floor()
        #print(cf)
        #Iterate through Price tree
        for col in reversed(range(0, len(tree)-1)):
            for row in range(0, col+1):
                rate = self.rates[row,col]
                pu = pd = 0.5
                tree[row,col] = np.exp(-1*rate*self.delta) * (pu*(cf[row,col+1]) \
                                                       + pd*(cf[row+1,col+1]))

        return (tree[0,0], tree)

In [5]:
zero_coupons = pd.read_csv('https://raw.githubusercontent.com/wrcarpenter/Interest-Rate-Models/main/Data/zcbs.csv') 
zcbs = zero_coupons.loc[zero_coupons['Date']=='3/8/2024']
zcbs = zcbs.drop("Date", axis=1)
zeros   = np.array(zcbs.iloc[:,0:60])
ho_lee = HoLeeModel(zeros,0.05,1/12)
rates = ho_lee.rateTree()

notion = 1000000
sigma = 0.05
strike = 5.50
delta = 1/12
cpn = 0
prob = 1/2

ir_cap = IRDerivativePricing(rates,prob,delta,"cap",notion,cpn,strike)
cap = ir_cap.priceTree()

ir_floor = IRDerivativePricing(rates,prob,delta,"floor",notion,cpn,strike)
floor = ir_floor.priceTree()

In [6]:
cap[0], floor[0]

(np.float64(610.7702387120615), np.float64(586.5325657734896))

In [7]:
pricing = np.zeros([20,6])
notion = 1000000
sigma = 0.05
strike = 3
delta = 1/12
cpn = 0
prob = 1/2

for row in range(pricing.shape[0]):
    pricing[row,0] = notion
    pricing[row,1] = strike
    pricing[row,2] = zeros.shape[1] #periods
    pricing[row,3] = sigma

    ir_cap = IRDerivativePricing(rates,prob,delta,"cap",notion,cpn,strike)
    cap = ir_cap.priceTree()
    
    ir_floor = IRDerivativePricing(rates,prob,delta,"floor",notion,cpn,strike)
    floor = ir_floor.priceTree()

    pricing[row, 4] = cap[0]
    pricing[row, 5] = floor[0]
    
    strike += 0.5

In [8]:
pricing = pd.DataFrame(pricing, columns=["Notional", "Strike", "Periods", "Volatility", "Cap Price","Floor Price"])

In [9]:
pricing

,Notional,Strike,Periods,Volatility,Cap Price,Floor Price
0,1000000.0,3.0,60.0,0.05,2098.026962,0.000000
1,1000000.0,3.5,60.0,0.05,1683.269104,0.000000
2,1000000.0,4.0,60.0,0.05,1268.511247,0.000000
3,1000000.0,4.5,60.0,0.05,1025.528097,171.774708
4,1000000.0,5.0,60.0,0.05,818.149168,379.153637
5,1000000.0,5.5,60.0,0.05,610.770239,586.532566
6,1000000.0,6.0,60.0,0.05,403.391310,793.911495
7,1000000.0,6.5,60.0,0.05,196.012381,1001.290424
8,1000000.0,7.0,60.0,0.05,0.000000,1220.035901
9,1000000.0,7.5,60.0,0.05,0.000000,1634.793759
